# BÁO CÁO TIẾN ĐỘ LÀM VIỆC — NGÀY 1: TIỀN XỬ LÝ & KỸ THUẬT TRÍCH CHỌN ĐẶC TRƯNG (FEATURE ENGINEERING)

## 🎯 MỤC TIÊU CỐT LÕI CỦA NGÀY 1
1. **Kiểm tra & thấu hiểu trọn vẹn bộ dữ liệu mới (657 dòng, 34 cột)**: Bao gồm dữ liệu khí tượng mặt đất trạm CMT8, mô hình ERA5 Open-Meteo, mô hình CAMS Copernicus, và các chuẩn Việt Nam (QĐ 1459/QĐ-TCMT, QCVN 05:2023/BTNMT).
2. **Thiết lập biến mục tiêu phân loại ($Y$ - Target)** chuẩn mực theo quy chuẩn Việt Nam: Biến nhị phân chính `kem_tro_len` ($VN\_AQI > 100$ tương đương $PM_{2.5} > 50\,\mu\text{g/m}^3$) và biến đa lớp `vn_aqi_cat`.
3. **Nhận diện và lên chiến lược xử lý bài toán Mất cân bằng dữ liệu (Class Imbalance)**: Lớp thiểu số chỉ chiếm ~7.0% (46/657 ngày).
4. **Thực hiện Kỹ thuật trích chọn đặc trưng (Feature Engineering)**: Tận dụng các biến mới có sẵn (`wind_sin`, `wind_cos`, `ngay_tu_mua`, `ky_nghi_le`, `tet`, `pbl_sang_m`, các khí CAMS) và tạo thêm đặc trưng trễ (Lags), độ trượt (Rolling stats), độ tù đọng khí quyển.
5. **Mã hóa (Encoding) & Chuẩn hóa thang đo (Feature Scaling)** cho các mô hình nhạy thang đo (đặc biệt là SVM).
6. **Chiến lược chia tập mẫu (Train/Test Split)** đúng quy chuẩn chuỗi thời gian (Time-based Split), ngăn ngừa hoàn toàn rò rỉ dữ liệu tương lai.
7. **Đóng gói tập dữ liệu sạch sẵn sàng huấn luyện** để chuyển sang Ngày 2.

## 📋 CHI TIẾT CÁC BƯỚC THỰC HIỆN KÈM HƯỚNG DẪN & GIẢI THÍCH

### BƯỚC 1.1: Tiếp nhận, Khám phá & Đánh giá chất lượng dữ liệu (Data Sanity Check)
* **Ý nghĩa:** Thành viên 1 bàn giao file `aqi_hcm_daily_v1.csv` gồm 657 ngày với 34 thuộc tính phong phú. Cần kiểm tra kỹ các nhóm biến:
  * Khí tượng trạm CMT8: `temperature`, `humidity`, `pm25`.
  * Khí tượng Open-Meteo: `wind`, `rain`, `pbl_sang_m` (lớp biên sáng sớm), `ap_suat`.
  * Hướng gió: `wind_dir_deg`, `wind_sin`, `wind_cos`, `wind_sector`.
  * Tích tụ khô hạn: `ngay_tu_mua` (số ngày liên tiếp không mưa).
  * Lịch & xã hội: `date`, `month`, `dow`, `mua`, `ky_nghi_le`, `tet`.
  * Khí CAMS: `co_cams`, `no_cams`, `no2_cams`, `so2_cams`, `o3_cams`, `pm10_cams`.
  * Chuẩn Việt Nam: `vn_aqi_pm25`, `vn_aqi_cat`, `qcvn_24h`, `vuot_qcvn`, `vuot_who`, `kem_tro_len`.

In [1]:
import pandas as pd
import numpy as np

# 1. Đọc dữ liệu mới
df = pd.read_csv(r"C:\Users\duong\projects\airvisual-data-collection\01_Data\aqi_hcm_daily.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Kích thước dữ liệu:", df.shape)
print("Khoảng thời gian:", df["date"].min().date(), "->", df["date"].max().date())
print("\nKiểm tra dữ liệu khuyết:")
null_summary = df.isnull().sum()
print(null_summary[null_summary > 0] if (null_summary > 0).any() else "Không có giá trị khuyết (Dữ liệu sạch hoàn hảo)!")

Kích thước dữ liệu: (650, 42)
Khoảng thời gian: 2024-11-26 -> 2026-10-19

Kiểm tra dữ liệu khuyết:
Không có giá trị khuyết (Dữ liệu sạch hoàn hảo)!


### BƯỚC 1.2: Định nghĩa Biến mục tiêu ($Y$ Target) & Nhận diện Mất cân bằng dữ liệu
* **Tại sao đổi từ `nang` (US EPA) sang `kem_tro_len` (QĐ 1459/QĐ-TCMT)?**
  * Trong bộ dữ liệu cũ, nhãn `nang` dùng ngưỡng Mỹ ($PM_{2.5} > 35.4\,\mu\text{g/m}^3$) dẫn tới 32% số ngày bị gắn nhãn ô nhiễm.
  * Trong bộ dữ liệu mới, nhóm chuyển sang áp dụng **Quy chuẩn chính thức của Việt Nam (QĐ 1459/QĐ-TCMT)**: Ngưỡng bắt đầu cảnh báo sức khỏe cho người dân là mức **Kém** ($PM_{2.5} > 50\,\mu\text{g/m}^3$ hay $VN\_AQI > 100$).
  * Do đó, biến mục tiêu nhị phân chuẩn là **`kem_tro_len`**:
    * $Y = 0$: Mức Tốt hoặc Trung bình ($VN\_AQI \le 100$).
    * $Y = 1$: Mức Kém, Xấu, Rất xấu ($VN\_AQI > 100$) — thời điểm bắt buộc các đơn vị sự kiện, du lịch và y tế phải kích hoạt phương án bảo vệ.
* **Thách thức mất cân bằng dữ liệu (Severe Class Imbalance):**
  * Lớp $1$ chỉ có 46 ngày (~7.0%), trong khi lớp $0$ có 611 ngày (~93.0%).
  * Nếu dùng mô hình thông thường dự đoán tất cả là 0, Accuracy vẫn đạt 93% nhưng vô dụng vì bỏ sót 100% các đợt ô nhiễm nguy hại.
  * Vì vậy, bắt buộc phải dùng `class_weight='balanced'`, tối ưu hóa ngưỡng cắt theo Youden Index, và ưu tiên chỉ số **Recall** và **PR-AUC**.

In [2]:
# Phân tích biến mục tiêu nhị phân kem_tro_len
print("Phân phối nhãn nhị phân 'kem_tro_len':")
print(df["kem_tro_len"].value_counts())
print(df["kem_tro_len"].value_counts(normalize=True) * 100)

# Phân tích biến phân loại đa lớp vn_aqi_cat
print("\nPhân phối nhãn đa lớp 'vn_aqi_cat':")
print(df["vn_aqi_cat"].value_counts())

Phân phối nhãn nhị phân 'kem_tro_len':
kem_tro_len
0    604
1     46
Name: count, dtype: int64
kem_tro_len
0    92.923077
1     7.076923
Name: proportion, dtype: float64

Phân phối nhãn đa lớp 'vn_aqi_cat':
vn_aqi_cat
Trung binh    429
Tot           175
Kem            45
Xau             1
Name: count, dtype: int64


### BƯỚC 1.3: Kỹ thuật trích chọn đặc trưng nâng cao (Feature Engineering)
Tận dụng các biến mới có sẵn kết hợp sinh thêm các đặc trưng động học khí quyển:

1. **Đặc trưng tự hồi quy (Lag Features):**
   * Nồng độ bụi mịn ngày hôm trước ($Lag_1$), 2 ngày trước ($Lag_2$), 3 ngày trước ($Lag_3$) phản ánh hiện tượng tích tụ và quán tính của các hạt aerosol trong khí quyển đô thị.
2. **Đặc trưng thống kê trượt (Rolling Windows):**
   * Trung bình trượt 3 ngày (`pm25_roll3_mean`), trung bình trượt 7 ngày (`pm25_roll7_mean`), độ biến thiên (`pm25_roll3_std`) tính hoàn toàn trên các ngày quá khứ.
3. **Đặc trưng tương tác và chỉ số khí quyển:**
   * `wind_rain_inter` = `wind` $\times$ `rain` (khi cả gió mạnh và mưa cùng xuất hiện, hiệu quả rửa trôi đạt cực đại).
   * `stagnation_ratio` = `ap_suat` / (`wind` + 0.1) (áp suất cao kết hợp đứng gió tạo điều kiện giữ bụi).
   * Tận dụng trực tiếp các biến có sẵn: `ngay_tu_mua`, `wind_sin`, `wind_cos`, `pbl_sang_m`, `no2_cams`, `co_cams`.

In [3]:
# 1. Đặc trưng trễ (Lags)
df["pm25_lag1"] = df["pm25"].shift(1)
df["pm25_lag2"] = df["pm25"].shift(2)
df["pm25_lag3"] = df["pm25"].shift(3)

# 2. Đặc trưng thống kê trượt (Rolling - tính trên quá khứ shift 1)
df["pm25_roll3_mean"] = df["pm25"].shift(1).rolling(window=3).mean()
df["pm25_roll7_mean"] = df["pm25"].shift(1).rolling(window=7).mean()
df["pm25_roll3_std"]  = df["pm25"].shift(1).rolling(window=3).std()

# 3. Đặc trưng tương tác vi khí hậu
df["wind_rain_inter"] = df["wind"] * df["rain"]
df["stagnation_ratio"] = df["ap_suat"] / (df["wind"] + 0.1)

# Loại bỏ các dòng đầu do shift(3) và rolling(7) tạo ra NaN
df_model = df.dropna().reset_index(drop=True)
print("Số dòng khả dụng sau Feature Engineering:", len(df_model))
df_model

Số dòng khả dụng sau Feature Engineering: 643


,date,pm25,pm25_cov,temperature,temperature_cov,humidity,humidity_cov,wind,rain,month,...,vuot_who,kem_tro_len,pm25_lag1,pm25_lag2,pm25_lag3,pm25_roll3_mean,pm25_roll7_mean,pm25_roll3_std,wind_rain_inter,stagnation_ratio
0,2024-12-03,53.9,100.0,28.5,100.0,59.3,100.0,5.0,4.3,12,...,1,1,62.0,64.8,45.5,57.433333,41.500000,10.428966,21.50,197.549020
1,2024-12-04,47.1,100.0,28.2,100.0,57.0,100.0,8.1,2.0,12,...,1,0,53.9,62.0,64.8,60.233333,43.500000,5.660683,16.20,122.841463
2,2024-12-05,43.7,100.0,28.0,100.0,57.3,100.0,12.5,1.2,12,...,1,0,47.1,53.9,62.0,54.333333,46.100000,7.459446,15.00,79.928571
3,2024-12-06,56.4,100.0,28.1,100.0,59.1,100.0,10.0,5.4,12,...,1,1,43.7,47.1,53.9,48.233333,49.642857,5.193586,54.00,99.752475
4,2024-12-07,38.1,100.0,28.3,100.0,58.9,100.0,7.3,3.2,12,...,1,0,56.4,43.7,47.1,49.066667,53.342857,6.574445,23.36,136.283784
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
638,2026-10-15,41.3,100.0,28.4,100.0,66.0,100.0,9.6,11.0,10,...,1,0,26.2,28.7,34.6,29.833333,28.971429,4.313158,105.60,104.164948
639,2026-10-16,38.7,100.0,29.1,100.0,62.3,100.0,11.6,0.3,10,...,1,0,41.3,26.2,28.7,32.066667,30.342857,8.093413,3.48,86.273504
640,2026-10-17,29.2,100.0,29.0,100.0,63.2,100.0,12.2,11.7,10,...,1,0,38.7,41.3,26.2,35.400000,30.871429,8.072794,142.74,82.105691
641,2026-10-18,26.4,100.0,28.3,100.0,65.3,100.0,10.8,5.0,10,...,1,0,29.2,38.7,41.3,36.400000,32.157143,6.369458,54.00,92.697248


### BƯỚC 1.4: Tuyển chọn đặc trưng & Chuẩn hóa thang đo (Feature Scaling)
Phân loại rõ danh sách đặc trưng nạp vào mô hình:

In [4]:
from sklearn.preprocessing import StandardScaler

# Nhóm biến số liên tục (Cần chuẩn hóa cho SVM và Logistics)
num_features = [
    "temperature", "humidity", "wind", "rain", "ap_suat", "pbl_sang_m",
    "wind_sin", "wind_cos", "ngay_tu_mua",
    "co_cams", "no2_cams", "so2_cams", "o3_cams", "pm10_cams",
    "pm25_lag1", "pm25_lag2", "pm25_lag3",
    "pm25_roll3_mean", "pm25_roll7_mean", "pm25_roll3_std",
    "wind_rain_inter", "stagnation_ratio"
]

# Nhóm biến nhị phân / chỉ số lịch (Không cần scale)
cat_features = [
    "month", "dow", "ky_nghi_le", "tet"
]

all_features = num_features + cat_features
print(f"Tổng số đặc trưng đầu vào cho mô hình: {len(all_features)}")

Tổng số đặc trưng đầu vào cho mô hình: 26


### BƯỚC 1.5: Chiến lược phân chia tập mẫu theo trục thời gian (Time-based Split)
* Dữ liệu từ 19/11/2024 đến 19/10/2026 (~650 ngày sạch).
* Bắt buộc dùng **Time-based Split** để không bị rò rỉ tương lai (Data Leakage):
  * Tập Train: 80% thời gian đầu (khoảng 520 ngày).
  * Tập Test: 20% thời gian cuối (khoảng 130 ngày).
* Kiểm tra tỷ lệ lớp dương tính `kem_tro_len = 1` ở cả Train và Test để đảm bảo tập test có đủ mẫu đánh giá.

In [5]:
split_idx = int(len(df_model) * 0.8)
train_df = df_model.iloc[:split_idx].copy()
test_df  = df_model.iloc[split_idx:].copy()

X_train_raw = train_df[all_features]
y_train = train_df["kem_tro_len"]

X_test_raw = test_df[all_features]
y_test = test_df["kem_tro_len"]

print(f"Tập Train: {len(train_df)} ngày ({train_df['date'].min().date()} -> {train_df['date'].max().date()})")
print(f"  Số ca ô nhiễm Kém trở lên (y=1): {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Tập Test : {len(test_df)} ngày ({test_df['date'].min().date()} -> {test_df['date'].max().date()})")
print(f"  Số ca ô nhiễm Kém trở lên (y=1): {y_test.sum()} ({y_test.mean()*100:.1f}%)")

# Chuẩn hóa thang đo đúng chuẩn chống rò rỉ: chỉ fit trên Train!
scaler = StandardScaler()
X_train_scaled = X_train_raw.copy()
X_test_scaled  = X_test_raw.copy()

X_train_scaled[num_features] = scaler.fit_transform(X_train_raw[num_features])
X_test_scaled[num_features]  = scaler.transform(X_test_raw[num_features])

Tập Train: 514 ngày (2024-12-03 -> 2026-06-12)
  Số ca ô nhiễm Kém trở lên (y=1): 41 (8.0%)
Tập Test : 129 ngày (2026-06-13 -> 2026-10-19)
  Số ca ô nhiễm Kém trở lên (y=1): 3 (2.3%)


### BƯỚC 1.6: Lưu trữ và Đóng gói kết quả Ngày 1

In [ ]:
import joblib

df_model.to_csv(r"C:\Users\duong\projects\airvisual-data-collection\01_Data\tv2_data_doc\classification_data_prepared.csv", index=False)

data_bundle = {
    "X_train_raw": X_train_raw,
    "X_test_raw": X_test_raw,
    "X_train_scaled": X_train_scaled,
    "X_test_scaled": X_test_scaled,
    "y_train": y_train,
    "y_test": y_test,
    "train_dates": train_df["date"],
    "test_dates": test_df["date"],
    "features_list": all_features,
    "num_features": num_features,
    "scaler": scaler
}
joblib.dump(data_bundle, r"C:\Users\duong\projects\airvisual-data-collection\01_Data\tv2_data_doc\prepared_ml_bundle.pkl")
print("Đã lưu prepared_ml_bundle.pkl thành công!")


Đã lưu prepared_ml_bundle.pkl thành công!
